# Enrichment Results

Summarizes ORA and BES outputs from `scripts/04_run_enrichment.py` across the feature-selection methods.

Reads artifacts under:
- `results/enrichment/<method>/summary.parquet`
- `results/enrichment/<method>/<library>/terms.parquet`
- `results/enrichment/<method>/<library>/bes_components.json`
- `results/enrichment/_null_cache/<library>/size_<N>_seed_<seed>.npz`

Methods are displayed in fixed order: **random, ttest, mi, rf_shap**.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "results").exists() and (cwd / "scripts").exists():
        return cwd
    if cwd.name == "notebooks" and (cwd.parent / "results").exists() and (cwd.parent / "scripts").exists():
        return cwd.parent
    raise FileNotFoundError(
        "Could not resolve repository root from current working directory. "
        "Open the notebook in the project workspace root or notebooks folder."
    )


REPO_ROOT = resolve_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.styles.colors import CATEGORICAL, PALETTE

ENRICHMENT_DIR = REPO_ROOT / "results" / "enrichment"
NULL_CACHE_DIR = ENRICHMENT_DIR / "_null_cache"
FIG_DIR = REPO_ROOT / "results" / "figs" / "comparison" / "enrichment"
FIG_DIR.mkdir(parents=True, exist_ok=True)

METHODS = ["random", "ttest", "mi", "rf_shap"]
METHOD_LABEL = {"ttest": "t-test", "random": "Random", "mi": "MI", "rf_shap": "RF SHAP"}

print(f"Repo root: {REPO_ROOT}")
print(f"Enrichment dir: {ENRICHMENT_DIR}")
print(f"Figure dir: {FIG_DIR}")


## Load Method Summaries And Metadata

In [ ]:
def load_enrichment_summaries(base_dir: Path) -> tuple[pd.DataFrame, dict[str, dict]]:
    if not base_dir.exists():
        raise FileNotFoundError(f"Enrichment directory not found: {base_dir}")

    summary_frames: list[pd.DataFrame] = []
    meta_by_method: dict[str, dict] = {}

    for method_dir in sorted(base_dir.iterdir()):
        if not method_dir.is_dir() or method_dir.name == "_null_cache":
            continue

        summary_path = method_dir / "summary.parquet"
        meta_path = method_dir / "meta.json"
        if not summary_path.exists():
            continue

        method = method_dir.name
        df = pd.read_parquet(summary_path).copy()
        df["method"] = method
        summary_frames.append(df)

        if meta_path.exists():
            meta_by_method[method] = json.loads(meta_path.read_text(encoding="utf-8"))

    if not summary_frames:
        raise FileNotFoundError(f"No summary.parquet files found under {base_dir}")

    summaries = pd.concat(summary_frames, ignore_index=True)
    return summaries, meta_by_method


summaries, meta_by_method = load_enrichment_summaries(ENRICHMENT_DIR)
required_cols = {"method", "library", "gene_list_size", "bes_raw", "bes_z", "bes_p_emp", "n_significant_terms"}
missing = required_cols.difference(summaries.columns)
if missing:
    raise ValueError(f"Missing expected summary columns: {sorted(missing)}")

method_order = [m for m in METHODS if m in summaries["method"].unique()]
summaries["method"] = pd.Categorical(summaries["method"], categories=method_order, ordered=True)
palette_method = {m: c for m, c in zip(method_order, CATEGORICAL[:len(method_order)])}

print(f"Methods loaded (in plot order): {method_order}")
print(f"Libraries loaded: {sorted(summaries['library'].unique().tolist())}")
print(f"Rows: {len(summaries):,}")

display(
    summaries.sort_values(["library", "method"])
    .reset_index(drop=True)
    .style.background_gradient(subset=["bes_raw", "bes_z"], cmap="YlGnBu")
    .background_gradient(subset=["bes_p_emp"], cmap="YlOrRd_r")
    .format({"bes_raw": "{:.3f}", "bes_z": "{:.3f}", "bes_p_emp": "{:.3f}"})
)


In [ ]:
method_coverage = (
    summaries.groupby("method", observed=False, as_index=False)
    .agg(
        n_libraries=("library", "nunique"),
        gene_list_size=("gene_list_size", "first"),
        total_significant_terms=("n_significant_terms", "sum"),
        max_bes_raw=("bes_raw", "max"),
        max_bes_z=("bes_z", "max"),
        min_p_emp=("bes_p_emp", "min"),
    )
)
method_coverage["method"] = pd.Categorical(method_coverage["method"], categories=method_order, ordered=True)
display(method_coverage.sort_values("method").reset_index(drop=True))

if meta_by_method:
    print("Meta overview")
    meta_rows = []
    for method in method_order:
        meta = meta_by_method.get(method, {})
        meta_rows.append({
            "method": method,
            "seed": meta.get("seed"),
            "b_perm": meta.get("b_perm"),
            "skip_null": meta.get("skip_null"),
            "background_size": meta.get("background_size"),
            "gene_list_size": meta.get("gene_list_size"),
        })
    display(pd.DataFrame(meta_rows))


## BES Comparison Plots

In [ ]:
FONTSIZE_TITLE   = 22
FONTSIZE_SUBPLOT = 20
FONTSIZE_LABEL   = 18
FONTSIZE_TICK    = 16
FONTSIZE_LEGEND  = 16

LIBRARY_LABEL = {
    "GO_Biological_Process_2025": "GO BP",
    "GO_Biological_Process_2023": "GO BP",
    "KEGG_2026":                  "KEGG",
    "KEGG_2021_Human":            "KEGG",
    "Reactome_Pathways_2024":     "Reactome",
    "Reactome_2022":              "Reactome",
}

plot_df = summaries.copy()
plot_df["library_short"] = plot_df["library"].map(
    lambda x: LIBRARY_LABEL.get(x, x.replace("_2023", "").replace("_2022", "").replace("_2021_Human", "").replace("_", " "))
)
library_order = (
    plot_df.drop_duplicates("library")
    .sort_values("library")["library_short"]
    .tolist()
)

fig, axes = plt.subplots(1, 3, figsize=(19, 5.0))

panels = [
    (axes[0], "bes_raw",   "Raw BES",            "BES",   None),
    (axes[1], "bes_z",     "BES z-score",        "z",     0.0),
    (axes[2], "bes_p_emp", "Empirical p-value",  "p_emp", 0.05),
]
for ax, ycol, title, ylabel, hline in panels:
    sns.barplot(
        data=plot_df, x="library_short", y=ycol,
        hue="method", hue_order=method_order, palette=palette_method,
        order=library_order, ax=ax,
    )
    ax.set_title(title, fontsize=FONTSIZE_SUBPLOT)
    ax.set_xlabel("Library", fontsize=FONTSIZE_LABEL)
    ax.set_ylabel(ylabel, fontsize=FONTSIZE_LABEL)
    ax.tick_params(axis="x", rotation=20, labelsize=FONTSIZE_TICK)
    ax.tick_params(axis="y", labelsize=FONTSIZE_TICK)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    if hline is not None:
        color = "red" if ycol == "bes_p_emp" else "black"
        ls = "--" if ycol == "bes_p_emp" else "-"
        ax.axhline(hline, color=color, linestyle=ls, linewidth=1.2, alpha=0.7)
    if ax.get_legend() is not None:
        ax.get_legend().remove()

handles = [plt.Rectangle((0, 0), 1, 1, color=palette_method[m]) for m in method_order]
labels = [METHOD_LABEL.get(m, m) for m in method_order]
fig.legend(
    handles, labels, title="Method",
    fontsize=FONTSIZE_LEGEND, title_fontsize=FONTSIZE_LEGEND,
    loc="upper center", bbox_to_anchor=(0.5, 0.02),
    ncol=len(method_order), frameon=True,
)

fig.suptitle("BES across libraries by method", fontsize=FONTSIZE_TITLE, y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "bes_comparison.pdf", bbox_inches="tight", dpi=300)
plt.show()

# Significant-terms heatmap
pivot_sig = (
    plot_df.pivot(index="library_short", columns="method", values="n_significant_terms")
    .reindex(index=library_order, columns=method_order)
)
pivot_sig.columns = [METHOD_LABEL.get(c, c) for c in pivot_sig.columns]

fig, ax = plt.subplots(figsize=(1.6 * len(method_order) + 3, 0.6 * len(library_order) + 2))
sns.heatmap(
    pivot_sig, annot=True, fmt=".0f",
    cmap="YlGnBu", linewidths=0.5, linecolor="white",
    annot_kws={"size": FONTSIZE_LABEL}, ax=ax,
    cbar_kws={"shrink": 0.85, "label": "n significant"},
)
ax.set_title("Significant terms by method × library", fontsize=FONTSIZE_TITLE)
ax.set_xlabel("Method", fontsize=FONTSIZE_LABEL)
ax.set_ylabel("Library", fontsize=FONTSIZE_LABEL)
ax.tick_params(axis="both", labelsize=FONTSIZE_TICK)
ax.collections[0].colorbar.ax.tick_params(labelsize=FONTSIZE_TICK)
fig.tight_layout()
fig.savefig(FIG_DIR / "n_significant_heatmap.pdf", bbox_inches="tight", dpi=300)
plt.show()


## Inspect Term Tables

Use the controls in the next cell to inspect the top-ranked terms (smallest q-values) for any method and library.

In [ ]:
SELECT_METHOD = method_order[0] if "ttest" not in method_order else "ttest"
SELECT_LIBRARY = sorted(summaries["library"].unique().tolist())[0]
TOP_N_TERMS = 20

terms_path = ENRICHMENT_DIR / SELECT_METHOD / SELECT_LIBRARY / "terms.parquet"
if not terms_path.exists():
    raise FileNotFoundError(f"Terms file not found: {terms_path}")

terms_df = pd.read_parquet(terms_path).copy()
terms_df = terms_df.sort_values(["q_value", "p_value", "term"], kind="mergesort")

print(f"Method:  {METHOD_LABEL.get(SELECT_METHOD, SELECT_METHOD)}")
print(f"Library: {SELECT_LIBRARY}")
print(f"Rows in term table: {len(terms_df):,}")

cols = [c for c in ["term", "q_value", "p_value", "overlap_size", "term_size", "genes"] if c in terms_df.columns]
top_terms = terms_df.loc[:, cols].head(TOP_N_TERMS).reset_index(drop=True)
display(
    top_terms.style
    .background_gradient(subset=["q_value", "p_value"], cmap="YlOrRd_r")
    .format({"q_value": "{:.4f}", "p_value": "{:.4f}"})
    .set_caption(f"Top {TOP_N_TERMS} terms — {METHOD_LABEL.get(SELECT_METHOD, SELECT_METHOD)} / {SELECT_LIBRARY}")
)


In [6]:
components_path = ENRICHMENT_DIR / SELECT_METHOD / SELECT_LIBRARY / "bes_components.json"
if not components_path.exists():
    raise FileNotFoundError(f"BES components file not found: {components_path}")

components_payload = json.loads(components_path.read_text(encoding="utf-8"))
core_fields = {
    "method": components_payload.get("method"),
    "library": components_payload.get("library"),
    "gene_list_size": components_payload.get("gene_list_size"),
    "background_size": components_payload.get("background_size"),
    "bes_raw": components_payload.get("bes_raw"),
    "bes_z": components_payload.get("bes_z"),
    "bes_p_emp": components_payload.get("bes_p_emp"),
    "null_effective": components_payload.get("null_effective"),
}

print("BES components summary")
display(pd.DataFrame([core_fields]))

term_contrib = components_payload.get("bes_components", {}).get("per_term_contributions", [])
if term_contrib:
    contrib_df = pd.DataFrame(term_contrib).sort_values("contribution", ascending=False)
    display(contrib_df.head(20))
else:
    print("No per-term contributions available (no significant terms under current q-threshold).")

BES components summary


,method,library,gene_list_size,background_size,bes_raw,bes_z,bes_p_emp,null_effective
0,random,GO_Biological_Process_2025,20,9511,0.0,-0.243646,1.0,500


No per-term contributions available (no significant terms under current q-threshold).


In [7]:
cache_rows = []
if NULL_CACHE_DIR.exists():
    for library_dir in sorted(NULL_CACHE_DIR.iterdir()):
        if not library_dir.is_dir():
            continue
        for npz_path in sorted(library_dir.glob("size_*_seed_*.npz")):
            data = np.load(npz_path)
            null_vals = data["null_values"]
            valid = null_vals[np.isfinite(null_vals)]
            cache_rows.append(
                {
                    "library": library_dir.name,
                    "cache_file": npz_path.name,
                    "n_total": int(len(null_vals)),
                    "n_valid": int(len(valid)),
                    "mean": float(valid.mean()) if len(valid) else np.nan,
                    "std": float(valid.std(ddof=0)) if len(valid) else np.nan,
                    "min": float(valid.min()) if len(valid) else np.nan,
                    "max": float(valid.max()) if len(valid) else np.nan,
                }
            )

cache_df = pd.DataFrame(cache_rows)
if cache_df.empty:
    print("No null-cache files found.")
else:
    display(cache_df.sort_values(["library", "cache_file"]).reset_index(drop=True))

,library,cache_file,n_total,n_valid,mean,std,min,max
0,GO_Biological_Process_2025,size_12_seed_42.npz,500,500,7.688379,24.489540,0.0,265.902050
1,GO_Biological_Process_2025,size_19_seed_42.npz,500,500,1.455331,6.676371,0.0,100.171916
2,GO_Biological_Process_2025,size_20_seed_42.npz,500,500,1.117347,4.585942,0.0,79.709570
3,GO_Biological_Process_2025,size_30_seed_42.npz,500,499,0.951097,2.764858,0.0,26.878865
4,KEGG_2026,size_12_seed_42.npz,500,500,0.558686,2.844089,0.0,37.822575
5,KEGG_2026,size_19_seed_42.npz,500,499,0.579847,7.091181,0.0,156.138186
6,KEGG_2026,size_20_seed_42.npz,500,500,0.460757,4.172109,0.0,87.007031
7,KEGG_2026,size_30_seed_42.npz,500,498,0.120988,0.652230,0.0,10.010762
8,Reactome_Pathways_2024,size_12_seed_42.npz,500,500,1.807559,16.067677,0.0,335.156255
9,Reactome_Pathways_2024,size_19_seed_42.npz,500,499,1.344805,6.648095,0.0,79.845573


## Gene-list provenance (full-data selection)

Each method's gene list now derives from `results/selection/<method>/full_data_ranking.parquet` via `scripts/04_run_enrichment.py`. The cell below surfaces the `gene_list.json` written alongside each enrichment run so we can see whether the list came from the `significant` mask or fell back to the top-`STABILITY_MIN_SIZE` ranking.

In [ ]:
gene_list_rows = []
for method in method_order:
    method_dir = ENRICHMENT_DIR / method
    gene_list_path = method_dir / "gene_list.json"
    if not gene_list_path.exists():
        continue
    payload = json.loads(gene_list_path.read_text(encoding="utf-8"))
    gene_list_rows.append({
        "method": METHOD_LABEL.get(method, method),
        "source": payload.get("source"),
        "min_size": payload.get("min_size"),
        "n_proteins": payload.get("n_proteins"),
        "n_genes": payload.get("n_genes"),
    })

if not gene_list_rows:
    print("No gene_list.json files found. Re-run scripts/04_run_enrichment.py to generate them.")
else:
    display(pd.DataFrame(gene_list_rows).reset_index(drop=True))
